# SAM3D Body Pipeline — B1 LiDAR tz Correction

Focused notebook for developing and evaluating LiDAR-based depth (tz) correction for SAM3D Body.
SAM3D Objects is not run here. The cyclist/motorcycle OBB merge is also omitted.

**B1**: After SAM3D Body inference, override each pedestrian's tz with the median LiDAR depth
of their clean in-mask surface points (filtered by HDBSCAN). tx and ty are recomputed accordingly.
All mesh vertices and joints are shifted by the depth correction. OBB is recomputed from shifted vertices.

Toggle `B1_LIDAR_CORRECTION = True/False` in the Config cell to compare baseline vs. B1.

## 0. Imports

In [13]:
import sys
import os
import shutil
import tempfile
from pathlib import Path

import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes

import plotly
import plotly.graph_objects as go


## 1. Config

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Dataset selector  →  change this one variable to switch datasets
# ══════════════════════════════════════════════════════════════════════════════
DATASET = 'ecp'           # 'ecp'  or  'nuscenes_mini'

if DATASET == 'nuscenes_mini':
    DATA_ROOT    = '/media/lleba/ECP_Nuscenes_01/nuScenes_mini'
    NUSCENES_VER = 'v1.0-mini'
    SCENE_IDX    = 6
    FRAME_IDX    = 20
elif DATASET == 'ecp':
    DATA_ROOT    = '/media/lleba/ECP_Nuscenes_01/output2/ecp2nuscenes'
    NUSCENES_VER = 'v1.0-trainval'
    SCENE_IDX    = 10
    FRAME_IDX    = 560
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}")

CAMERA = 'CAM_FRONT'

DEV_ROOT = Path('/home/lleba/Thesis/Development')

from huggingface_hub import hf_hub_download
SAM3_CKPT = hf_hub_download(repo_id='facebook/sam3', filename='sam3.pt')

SAM3D_BODY_REPO = DEV_ROOT / 'Models' / 'SAM3D' / 'sam-3d-body'
SAM3D_BODY_CKPT = SAM3D_BODY_REPO / 'checkpoints' / 'sam-3d-body-dinov3' / 'model.ckpt'
SAM3D_BODY_MHR  = SAM3D_BODY_REPO / 'checkpoints' / 'sam-3d-body-dinov3' / 'assets' / 'mhr_model.pt'

SAM3_PROMPTS = {
    'pedestrian': 'body',
}

SAM3_SCORE_THRESH     = 0.5
MIN_MASK_PX           = 100
BODY_BBOX_THRESH      = 0.8
SAM3_IOU_MERGE_THRESH = 0.4

# ── B1 Stage 1: LiDAR tz correction ──────────────────────────────────────────
# When True: override each pedestrian's tz with HDBSCAN-filtered median LiDAR depth.
# tx/ty recomputed from mask centroid + new tz. Vertices shifted; joints_3d NOT shifted.
# When False: baseline SAM3D Body tz (appearance-based scale prediction).
B1_LIDAR_CORRECTION = True

# ── B1 Stage 2: Ground anchoring ─────────────────────────────────────────────
# When True: fit PseudoLabeler MLP on the raw LiDAR sweep, then query the fitted
# surface at each pedestrian's (x,y) to anchor mesh feet to the ground.
# Falls back to global RANSAC if PseudoLabeler fitting fails.
B2_GROUND_ANCHORING = True

# ── LiDAR sweep aggregation ───────────────────────────────────────────────────
# Number of extra sweeps to include before/after the anchor frame (~50-70 ms apart).
# Set both to 0 for single-frame mode (original behaviour).
#
# HDBSCAN + median path  : uses all aggregated sweeps.
#   Motion spread (~0.1-0.3 m for walking) is small vs. body depth (~0.5 m);
#   median is symmetric so past/future offsets cancel.
# 15th-percentile fallback: anchor sweep only.
#   Displaced non-anchor points of a moving pedestrian can bias the front-surface
#   estimate, so only anchor-frame points are used there.
# B2 ground fitting (PseudoLabeler / RANSAC): uses all aggregated sweeps.
#   Ground is static; more points = better plane fit.
N_BEFORE = 0   # sweeps before anchor
N_AFTER  = 0   # sweeps after anchor

# ── HDBSCAN preset selector ───────────────────────────────────────────────────
# NOTE: Hull anchoring (dense-depth clamping via HDBSCAN min-depth) is ONLY used
#       in SAM3D Objects (O4/O5 CFormer pipeline) and does NOT exist in this notebook.
#       To investigate hull anchoring, run the full pipeline via autolabeling_pipeline.ipynb
#       with hull_anchoring: true/false in configs/nuscenes.yaml.
#
# True  → OLD logic: always pick the largest HDBSCAN cluster (~300 TPs on nuScenes mini)
# False → NEW logic: proximity-aware selection — when two clusters are roughly equal in size,
#                    prefer the one whose centroid is closer to the ego vehicle origin
USE_OLD_HDBSCAN = True

_HDBSCAN_OLD = dict(
    min_cluster_size  = 3,
    min_samples       = 1,
    cluster_eps       = 0.20,   # metres — compact pedestrian body surface
    proximity_min_pts = 30,     # not used in old selection logic, kept for API compatibility
    proximity_ratio   = 0.70,   # not used in old selection logic, kept for API compatibility
)
_HDBSCAN_NEW = dict(
    min_cluster_size  = 3,
    min_samples       = 1,
    cluster_eps       = 0.20,
    proximity_min_pts = 30,     # min pts in largest cluster to trigger proximity check
    proximity_ratio   = 0.70,   # second/largest ratio threshold for proximity selection
)

_hdbscan_preset      = _HDBSCAN_OLD if USE_OLD_HDBSCAN else _HDBSCAN_NEW
HDBSCAN_MIN_CLUSTER_SIZE = _hdbscan_preset['min_cluster_size']
HDBSCAN_MIN_SAMPLES      = _hdbscan_preset['min_samples']
HDBSCAN_CLUSTER_EPS      = _hdbscan_preset['cluster_eps']
PROXIMITY_MIN_PTS        = _hdbscan_preset['proximity_min_pts']
PROXIMITY_RATIO          = _hdbscan_preset['proximity_ratio']

# ── Ego-body exclusion zone ───────────────────────────────────────────────────
# Removes ego roof/windshield reflections that form a dense strip at z≈sensor
# height and can dominate HDBSCAN in-mask clustering after sweep aggregation.
USE_EGO_BODY_FILTER = True   # set False to disable
EGO_BOX_HALF_X = 4.0   # half-length fore/aft [m]
EGO_BOX_HALF_Y = 1.5   # half-width left/right [m]
EGO_BOX_Z_MIN  = 0.5   # lower Z cutoff — preserves close ground-level returns [m]
EGO_BOX_Z_MAX  = 2.5   # upper Z cutoff — preserves very tall objects above [m]

# ── Max LiDAR range filter ────────────────────────────────────────────────────
# Removes points beyond this BEV distance from ego (metres).
# Set +2 m over the 50 m evaluation limit to cut distant background returns
# that could form spurious HDBSCAN clusters inside near-object masks.
MAX_LIDAR_RANGE_M = 52.0


print(f'Dataset: {DATASET}  |  scene={SCENE_IDX}  frame={FRAME_IDX}  cam={CAMERA}')
print(f'B1={B1_LIDAR_CORRECTION}  B2={B2_GROUND_ANCHORING}')
print(f'LiDAR aggregation: {N_BEFORE} before + 1 anchor + {N_AFTER} after = {N_BEFORE + 1 + N_AFTER} sweeps')
hdbscan_label = 'OLD (largest cluster)' if USE_OLD_HDBSCAN else 'NEW (proximity-aware)'
print(f'HDBSCAN preset: {hdbscan_label}  '
      f'(min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}  cluster_eps={HDBSCAN_CLUSTER_EPS}m)')
print(f'Ego-body filter: {"ON" if USE_EGO_BODY_FILTER else "OFF"}  (|X|<{EGO_BOX_HALF_X}m, |Y|<{EGO_BOX_HALF_Y}m, Z∈[{EGO_BOX_Z_MIN}, {EGO_BOX_Z_MAX}]m)')
print(f'Max range filter: {MAX_LIDAR_RANGE_M} m (BEV distance from ego)')
print(f'SAM3 checkpoint: {SAM3_CKPT}')

## 2. Path Setup

In [ ]:
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_float32_matmul_precision('high')

sys.path.insert(0, str(SAM3D_BODY_REPO))

nusc = NuScenes(version=NUSCENES_VER, dataroot=DATA_ROOT, verbose=False)
scenes = nusc.scene
scene  = scenes[SCENE_IDX]

# Walk to the requested frame
cam_sd_token = nusc.get('sample', scene['first_sample_token'])['data'][CAMERA]
for _ in range(FRAME_IDX):
    sd = nusc.get('sample_data', cam_sd_token)
    cam_sd_token = sd['next']
    if not cam_sd_token:
        raise ValueError(f'FRAME_IDX={FRAME_IDX} exceeds scene length.')

print(f'Device: {device}')
print(f'Scene: {scene["name"]}  ({SCENE_IDX=})')
print(f'Camera sample_data token: {cam_sd_token}')


## 3. Data Loading

In [ ]:
def load_frame(nusc, scene_idx, frame_idx, camera='CAM_FRONT'):
    """
    Load image, intrinsics, and camera-to-ego extrinsics for one frame.

    Returns
    -------
    img_bgr       : (H, W, 3) uint8 BGR image
    img_rgb       : (H, W, 3) uint8 RGB image
    K             : (3, 3) float64 camera intrinsics
    R_c2e         : (3, 3) float64 rotation camera → ego
    t_c2e         : (3,)   float64 translation camera → ego
    img_path      : str    absolute path to image file
    cam_sd_token  : str    nuScenes sample_data token for this camera frame
    """
    scene   = nusc.scene[scene_idx]
    token   = nusc.get('sample', scene['first_sample_token'])['data'][camera]
    for _ in range(frame_idx):
        rec   = nusc.get('sample_data', token)
        token = rec['next']
        assert token, f'frame_idx {frame_idx} exceeds scene length'

    rec      = nusc.get('sample_data', token)
    img_path = nusc.get_sample_data_path(token)
    cal      = nusc.get('calibrated_sensor', rec['calibrated_sensor_token'])

    K     = np.array(cal['camera_intrinsic'], dtype=np.float64)
    R_c2e = Quaternion(cal['rotation']).rotation_matrix.astype(np.float64)
    t_c2e = np.array(cal['translation'], dtype=np.float64)

    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_rgb, img_bgr, K, R_c2e, t_c2e, img_path, token


nusc    = NuScenes(version=NUSCENES_VER, dataroot=DATA_ROOT, verbose=False)
img_rgb, img_bgr, K, R_c2e, t_c2e, img_path, cam_sd_token = load_frame(
    nusc, SCENE_IDX, FRAME_IDX, CAMERA
)
H, W = img_rgb.shape[:2]
fx, fy, cx, cy = K[0,0], K[1,1], K[0,2], K[1,2]

print(f'Image: {img_path}')
print(f'  Size : {W}×{H}')
print(f'  fx={fx:.1f}  fy={fy:.1f}  cx={cx:.1f}  cy={cy:.1f}')

plt.figure(figsize=(14, 5))
plt.imshow(img_rgb)
plt.axis('off')
plt.title(f'Scene {SCENE_IDX}, frame {FRAME_IDX}, {CAMERA}')
plt.tight_layout()
plt.show()


## 3.5 LiDAR Loading, Ego-Motion Compensation, and Sweep Aggregation

In [ ]:
# ── Find anchor LiDAR sweep ──────────────────────────────────────────────────
cam_sd = nusc.get('sample_data', cam_sd_token)
cam_ts = cam_sd['timestamp']

# Anchor ego pose used as the reference frame for ego-motion compensation
cam_ep       = nusc.get('ego_pose', cam_sd['ego_pose_token'])
R_e2g_anchor = Quaternion(cam_ep['rotation']).rotation_matrix.astype(np.float64)
t_e2g_anchor = np.array(cam_ep['translation'], dtype=np.float64)

if cam_sd['is_key_frame']:
    _sample     = nusc.get('sample', cam_sd['sample_token'])
    lidar_token = _sample['data']['LIDAR_TOP']
else:
    lidar_token = min(
        (sd for sd in nusc.sample_data if sd['channel'] == 'LIDAR_TOP'),
        key=lambda sd: abs(sd['timestamp'] - cam_ts),
    )['token']

# Keep anchor LiDAR calibration available for downstream cells
lidar_sd  = nusc.get('sample_data', lidar_token)
lidar_cal = nusc.get('calibrated_sensor', lidar_sd['calibrated_sensor_token'])
R_l2e     = Quaternion(lidar_cal['rotation']).rotation_matrix.astype(np.float64)
t_l2e     = np.array(lidar_cal['translation'], dtype=np.float64)

# ── Walk linked list to collect ±N sweeps ────────────────────────────────────
# Duplicate file entries (an ECP conversion artefact where multiple tokens point
# to the same .bin) are silently skipped so they never consume a slot.

def _walk_lidar_tokens(nusc, anchor_token, n_before, n_after):
    anchor_path = nusc.get_sample_data_path(anchor_token)
    seen_paths  = {anchor_path}

    backward = []
    tok = nusc.get('sample_data', anchor_token)['prev']
    while tok and len(backward) < n_before:
        path = nusc.get_sample_data_path(tok)
        if path not in seen_paths:
            backward.append(tok)
            seen_paths.add(path)
        tok = nusc.get('sample_data', tok)['prev']

    tokens = [(-(len(backward) - i), t) for i, t in enumerate(reversed(backward))]
    tokens.append((0, anchor_token))

    count = 0
    tok   = nusc.get('sample_data', anchor_token)['next']
    while tok and count < n_after:
        path = nusc.get_sample_data_path(tok)
        if path not in seen_paths:
            count += 1
            tokens.append((count, tok))
            seen_paths.add(path)
        tok = nusc.get('sample_data', tok)['next']

    return tokens


sweep_tokens = _walk_lidar_tokens(nusc, lidar_token, N_BEFORE, N_AFTER)
print(f'Collecting {len(sweep_tokens)} LiDAR sweep(s)  '
      f'({N_BEFORE} before + 1 anchor + {N_AFTER} after):')

# ── Load & ego-motion compensate all sweeps ───────────────────────────────────
# Each sweep is transformed to the anchor ego frame:
#   p_lidar -> p_ego_sweep = R_l2e @ p + t_l2e
#   p_global = R_e2g_sweep @ p_ego_sweep + t_e2g_sweep
#   p_ego_anchor = R_e2g_anchor.T @ (p_global - t_e2g_anchor)

all_pts_ego = []   # one (N_i, 3) float32 array per sweep, anchor ego frame
anchor_pts  = None # anchor-sweep-only cloud (for 15th-pct fallback in B1)

for rel_idx, tok in sweep_tokens:
    sd       = nusc.get('sample_data', tok)
    lid_path = nusc.get_sample_data_path(tok)
    lid_cal  = nusc.get('calibrated_sensor', sd['calibrated_sensor_token'])
    ego_pose = nusc.get('ego_pose', sd['ego_pose_token'])

    R_l2e_sw = Quaternion(lid_cal['rotation']).rotation_matrix.astype(np.float64)
    t_l2e_sw = np.array(lid_cal['translation'], dtype=np.float64)
    R_e2g_sw = Quaternion(ego_pose['rotation']).rotation_matrix.astype(np.float64)
    t_e2g_sw = np.array(ego_pose['translation'], dtype=np.float64)

    pts_raw     = np.fromfile(lid_path, dtype=np.float32).reshape(-1, 5)[:, :3].astype(np.float64)
    pts_ego_sw  = (R_l2e_sw @ pts_raw.T).T + t_l2e_sw           # LiDAR -> sweep ego
    # ── Ego-body filter in sweep's own ego frame (vehicle always at origin) ──
    if USE_EGO_BODY_FILTER:
        _sw_box = (
            (np.abs(pts_ego_sw[:, 0]) < EGO_BOX_HALF_X) &
            (np.abs(pts_ego_sw[:, 1]) < EGO_BOX_HALF_Y) &
            (pts_ego_sw[:, 2] > EGO_BOX_Z_MIN) &
            (pts_ego_sw[:, 2] < EGO_BOX_Z_MAX)
        )
        pts_ego_sw = pts_ego_sw[~_sw_box]
    pts_global  = (R_e2g_sw @ pts_ego_sw.T).T + t_e2g_sw        # sweep ego -> global
    pts_ego_anc = (R_e2g_anchor.T @ (pts_global - t_e2g_anchor).T).T  # global -> anchor ego

    pts_f32 = pts_ego_anc.astype(np.float32)
    all_pts_ego.append(pts_f32)

    dt_ms = (sd['timestamp'] - cam_ts) / 1000.0
    tag   = '  <- anchor' if rel_idx == 0 else ''
    print(f'  [{rel_idx:+d}]  {len(pts_f32):6,} pts  dt={dt_ms:+7.1f} ms{tag}')

    if rel_idx == 0:
        anchor_pts = pts_f32

# Full aggregated cloud: used by B1 HDBSCAN + median, and all of B2
pts_ego = np.concatenate(all_pts_ego, axis=0)
print(f'\nAggregated : {len(pts_ego):,} pts from {len(sweep_tokens)} sweep(s)')
print(f'Anchor-only: {len(anchor_pts):,} pts  (used for 15th-pct fallback in B1)')

# Ego-body filter applied per-sweep above (in each sweep's own ego frame).
print(f'Ego-body filter: applied per-sweep ({"ON" if USE_EGO_BODY_FILTER else "OFF"})')

# ── Max range filter ──────────────────────────────────────────────────────────
_bev_agg   = np.sqrt(pts_ego[:, 0]**2 + pts_ego[:, 1]**2)
_n_agg     = len(pts_ego)
pts_ego    = pts_ego[_bev_agg <= MAX_LIDAR_RANGE_M]
_bev_anc   = np.sqrt(anchor_pts[:, 0]**2 + anchor_pts[:, 1]**2)
anchor_pts = anchor_pts[_bev_anc <= MAX_LIDAR_RANGE_M]
print(f'Range filter ({MAX_LIDAR_RANGE_M} m): removed {_n_agg - len(pts_ego):,} agg pts → {len(pts_ego):,} remaining')

# ── Project into anchor camera frame ─────────────────────────────────────────
# Called twice: once for the full aggregated cloud, once for the anchor-only cloud.

def _project_to_camera(pts, R_c2e, t_c2e, fx, fy, cx, cy, H, W):
    pts_cam = (R_c2e.T @ (pts.astype(np.float64) - t_c2e).T).T
    Z_all   = pts_cam[:, 2]
    front   = Z_all > 0
    u_all   = pts_cam[front, 0] / Z_all[front] * fx + cx
    v_all   = pts_cam[front, 1] / Z_all[front] * fy + cy
    in_img  = (u_all >= 0) & (u_all < W) & (v_all >= 0) & (v_all < H)
    return (pts[front][in_img].astype(np.float32),
            u_all[in_img].astype(np.float32),
            v_all[in_img].astype(np.float32),
            pts_cam[front][in_img][:, 2].astype(np.float32))

# Aggregated: used for HDBSCAN + median in B1, and for the LiDAR overlay viz
pts_ego_vis, u_vis, v_vis, Z_vis = _project_to_camera(
    pts_ego, R_c2e, t_c2e, fx, fy, cx, cy, H, W
)
# Anchor-only: used for 15th-percentile fallback in B1
pts_ego_vis_anchor, u_vis_anchor, v_vis_anchor, Z_vis_anchor = _project_to_camera(
    anchor_pts, R_c2e, t_c2e, fx, fy, cx, cy, H, W
)

print(f'\nVisible in {CAMERA}:')
print(f'  Aggregated : {len(pts_ego_vis):,} pts  '
      f'(depth {Z_vis.min():.1f}-{Z_vis.max():.1f} m)')
print(f'  Anchor-only: {len(pts_ego_vis_anchor):,} pts  '
      f'(depth {Z_vis_anchor.min():.1f}-{Z_vis_anchor.max():.1f} m)')

In [ ]:
# ── 3.5.1 LiDAR points projected onto image ──────────────────────────────────
# Points are coloured by depth (Z_vis, metres). Closer = warm, farther = cool.

_depth_norm = plt.Normalize(vmin=Z_vis.min(), vmax=min(Z_vis.max(), 80.0))
_cmap       = plt.cm.plasma

fig, ax = plt.subplots(figsize=(16, 6))
ax.imshow(img_rgb)
sc = ax.scatter(
    u_vis, v_vis,
    c=Z_vis, cmap=_cmap, norm=_depth_norm,
    s=2, linewidths=0, alpha=0.85,
)
cbar = fig.colorbar(sc, ax=ax, fraction=0.015, pad=0.01)
cbar.set_label('Depth (m)', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

ax.axis('off')
ax.set_title(
    f'LiDAR projected onto {CAMERA}  —  {len(u_vis):,} pts  '
    f'(depth {Z_vis.min():.1f}–{Z_vis.max():.1f} m)',
    color='white',
)
fig.patch.set_facecolor('#111111')
ax.set_facecolor('#111111')
plt.tight_layout()
plt.show()


In [ ]:
# ── 3.6 PseudoLabeler — Ground Surface Fitting ───────────────────────────────
# PseudoLabeler fits a small MLP  gθ: R² → R  (x,y) → z  directly on the raw
# LiDAR sweep using an asymmetric loss that finds the ground surface without
# needing per-point ground labels.  Once fitted, the MLP can be queried at any
# (x,y) — including locations with no LiDAR returns — giving a smooth,
# continuous ground height estimate.
#
# No subprocess, no separate conda env: PseudoLabeler is pure PyTorch and runs
# in this kernel.  RANSAC is kept only as an emergency fallback.

import sys as _sys
_pl_path = str(DEV_ROOT / 'Models' / 'TerraSeg' / 'PseudoLabeler_scripts')
if _pl_path not in _sys.path:
    _sys.path.insert(0, _pl_path)
from pseudolabeler_model import PseudoLabeler
from pseudolabeler_loss  import pseudolabeler_loss as _pl_loss

pseudolabeler_model = None   # set below if B2_GROUND_ANCHORING=True
_device_pl = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if B2_GROUND_ANCHORING:
    pl_model = PseudoLabeler().to(_device_pl)
    pl_model.train()

    # Preprocess: strip ego vehicle points, remove bottom noise
    _pc = torch.tensor(pts_ego, dtype=torch.float32).to(_device_pl)
    _pc = pl_model.remove_ego_points(_pc)
    _pc = pl_model.preprocess_denoise_pc(_pc)
    print(f'PseudoLabeler fitting on {len(_pc):,} points...')

    _opt  = torch.optim.AdamW(pl_model.parameters(), lr=1e-2, weight_decay=1e-4)
    _sch  = torch.optim.lr_scheduler.ReduceLROnPlateau(_opt, mode='min', factor=0.5, patience=50)

    _best_loss, _best_state, _no_improve, _ema = float('inf'), None, 0, None

    for _step in range(2500):
        _opt.zero_grad()
        _pred = pl_model(_pc)
        _loss = _pl_loss(_pred, _pc[:, 2])
        _loss.backward()
        torch.nn.utils.clip_grad_norm_(pl_model.parameters(), 5.0)
        _opt.step()
        _sch.step(_loss.item())

        _ema = _loss.item() if _ema is None else 0.9 * _ema + 0.1 * _loss.item()

        if _ema < _best_loss:
            _best_loss  = _ema
            _best_state = {k: v.detach().cpu() for k, v in pl_model.state_dict().items()}
            _no_improve = 0
        else:
            _no_improve += 1
            if _no_improve >= 150:
                print(f'  Early stopping at step {_step + 1}  (best loss={_best_loss:.5f})')
                break

    if _best_state:
        pl_model.load_state_dict(_best_state)
    pl_model.eval().to(_device_pl)
    pseudolabeler_model = pl_model
    print(f'PseudoLabeler ready  ({_step + 1} steps, best loss={_best_loss:.5f})')
else:
    print('B2_GROUND_ANCHORING=False — PseudoLabeler skipped.')

## 4. SAM3 Segmentation

In [ ]:
from sam3.model_builder import build_sam3_video_predictor

print('Loading SAM3...')
gpus = range(torch.cuda.device_count())
sam3_predictor = build_sam3_video_predictor(gpus_to_use=gpus, checkpoint_path=str(SAM3_CKPT))
print('  SAM3 ready.')


def run_sam3_single_frame(predictor, img_rgb, text_prompt, score_thresh=0.5, min_mask_px=100):
    """
    Run SAM3 with a text prompt on a single frame.

    The image is written to a temp directory as a single-frame 'video',
    then SAM3 propagates over it and the session is closed.

    Parameters
    ----------
    predictor    : SAM3 video predictor instance
    img_rgb      : (H, W, 3) uint8 RGB image
    text_prompt  : str  e.g. 'car', 'pedestrian', 'bicycle and cyclist'
    score_thresh : float  min object probability to keep
    min_mask_px  : int    min mask area to keep

    Returns
    -------
    detections : list of dicts, each with:
        'binary_mask'  : (H, W) bool
        'score'        : float
        'box_xywh'     : (4,) float  normalised [x, y, w, h]
        'prompt'       : str
    """
    tmp_dir = tempfile.mkdtemp(prefix='sam3_frame_')
    try:
        Image.fromarray(img_rgb).save(os.path.join(tmp_dir, '00000.jpg'))
        resp       = predictor.handle_request(dict(type='start_session', resource_path=tmp_dir))
        session_id = resp['session_id']
        predictor.handle_request(dict(
            type='add_prompt', session_id=session_id, frame_index=0, text=text_prompt
        ))
        frame_outputs = {}
        with torch.autocast('cuda', dtype=torch.bfloat16):
            for resp in predictor.handle_stream_request(dict(
                type='propagate_in_video', session_id=session_id
            )):
                frame_outputs[resp['frame_index']] = resp['outputs']
        predictor.handle_request(dict(type='close_session', session_id=session_id))
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    out      = frame_outputs.get(0, {})
    obj_ids  = out.get('out_obj_ids', [])
    probs    = np.asarray(out.get('out_probs', []), dtype=np.float32)
    masks    = out.get('out_binary_masks', [])
    boxes    = np.asarray(out.get('out_boxes_xywh', []), dtype=np.float32)

    detections = []
    for i, (oid, prob) in enumerate(zip(obj_ids, probs)):
        if prob < score_thresh:
            continue
        bm = np.asarray(masks[i], dtype=bool)
        if bm.sum() < min_mask_px:
            continue
        detections.append({
            'binary_mask': bm,
            'score':       float(prob),
            'box_xywh':    boxes[i] if len(boxes) > i else None,
            'prompt':      text_prompt,
        })
    return detections


# Run SAM3 for all classes
sam3_results = {}
for prompt, pipeline in SAM3_PROMPTS.items():
    print(f'Running SAM3: "{prompt}" ...', end=' ')
    dets = run_sam3_single_frame(
        sam3_predictor, img_rgb, prompt,
        score_thresh=SAM3_SCORE_THRESH, min_mask_px=MIN_MASK_PX
    )
    sam3_results[prompt] = dets
    print(f'{len(dets)} detection(s)')

total = sum(len(v) for v in sam3_results.values())
print(f'\nTotal detections: {total}')

# ── Cross-class deduplication ───────────────────────────────────────────────
CROSS_CLASS_PRIORITY = {
    'pedestrian':           0,
    'car':                  1,
    'bicycle and cyclist':  2,
    'truck':                3,
    'motorcycle':           4,
    'trailer':              5,
    'bus':                  6,
    'construction vehicle': 7,
}
CROSS_CLASS_IOU_THRESH = 0.5


def dedup_cross_class(results, iou_thresh=CROSS_CLASS_IOU_THRESH, priority=CROSS_CLASS_PRIORITY):
    flat = [(p, d) for p, dets in results.items() for d in dets]
    keep = [True] * len(flat)
    for i in range(len(flat)):
        for j in range(i + 1, len(flat)):
            if not keep[i] or not keep[j]:
                continue
            p_i, d_i = flat[i]
            p_j, d_j = flat[j]
            if p_i == p_j:
                continue
            a, b  = d_i['binary_mask'], d_j['binary_mask']
            inter = np.logical_and(a, b).sum()
            if inter == 0:
                continue
            iou = inter / np.logical_or(a, b).sum()
            if iou < iou_thresh:
                continue
            if priority.get(p_i, 0) >= priority.get(p_j, 0):
                keep[j] = False
            else:
                keep[i] = False

    out = {p: [] for p in results}
    for (p, d), k in zip(flat, keep):
        if k:
            out[p].append(d)

    removed = sum(1 for k in keep if not k)
    print(f'Cross-class dedup: removed {removed} duplicate mask(s)  (IoU thresh={iou_thresh})')
    for p, dets in out.items():
        if dets:
            print(f'  {p}: {len(dets)}')
    return out


sam3_results = dedup_cross_class(sam3_results)
total_after  = sum(len(v) for v in sam3_results.values())
print(f'Detections after dedup: {total_after}')

# ── Class colour map (one fixed colour per prompt) ────────────────────────
CLASS_COLORS = {
    'pedestrian':           np.array([255,  60,  60], dtype=np.float32),
    'car':                  np.array([ 60, 180,  60], dtype=np.float32),
    'bicycle and cyclist':  np.array([ 60, 120, 255], dtype=np.float32),
    'motorcycle':           np.array([255, 160,   0], dtype=np.float32),
    'bus':                  np.array([180,  60, 255], dtype=np.float32),
    'truck':                np.array([  0, 210, 210], dtype=np.float32),
    'construction vehicle': np.array([255,  80, 180], dtype=np.float32),
    'trailer':              np.array([140, 200,  60], dtype=np.float32),
}
OUTLINE_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))  # outline thickness

overlay = img_rgb.copy().astype(np.float32)
outline_layer = np.zeros((H, W), dtype=bool)   # accumulate all outlines

# First pass: fill colours
for prompt, dets in sam3_results.items():
    col = CLASS_COLORS.get(prompt, np.array([200, 200, 200], dtype=np.float32))
    for d in dets:
        mask = d['binary_mask']
        overlay[mask] = overlay[mask] * 0.45 + col * 0.55

# Second pass: compute and draw outlines (dilated mask XOR original = boundary ring)
for prompt, dets in sam3_results.items():
    for d in dets:
        mask_u8  = d['binary_mask'].astype(np.uint8) * 255
        dilated  = cv2.dilate(mask_u8, OUTLINE_KERNEL)
        outline  = (dilated > 0) & ~d['binary_mask']
        outline_layer |= outline

# Draw outlines in white on top of colour overlay
overlay[outline_layer] = np.array([255, 255, 255], dtype=np.float32)

# Build legend (one entry per class that has detections)
legend_patches = []
for prompt, dets in sam3_results.items():
    if not dets:
        continue
    col = CLASS_COLORS.get(prompt, np.array([200, 200, 200], dtype=np.float32))
    legend_patches.append(
        patches.Patch(color=col / 255, label=f'{prompt} ({len(dets)})')
    )

fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.imshow(overlay.astype(np.uint8))
ax.axis('off')
ax.set_title('SAM3 detections (all classes)')
if legend_patches:
    ax.legend(handles=legend_patches, loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

# Unload SAM3 before loading the next model
del sam3_predictor
gc.collect()
torch.cuda.empty_cache()
print(f'SAM3 unloaded. GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')


## 5. SAM3D Body — Pedestrian Meshes

In [ ]:
from sam_3d_body import load_sam_3d_body, SAM3DBodyEstimator
from tools.build_detector import HumanDetector

# SAM3 loading leaves torch.autocast globally enabled (BF16).
# Explicitly disable it before loading SAM3D Body so model init and inference
# are not affected. process_one_image is also wrapped below for safety.
print(f'Autocast state inherited from SAM3: {torch.is_autocast_enabled()}')
torch.autocast('cuda', enabled=False).__enter__()
print(f'Autocast disabled: {torch.is_autocast_enabled()}')

print('Loading SAM3D Body...')
_body_model, _body_cfg = load_sam_3d_body(
    str(SAM3D_BODY_CKPT), device=device, mhr_path=str(SAM3D_BODY_MHR)
)
body_estimator = SAM3DBodyEstimator(
    sam_3d_body_model=_body_model,
    model_cfg=_body_cfg,
    human_detector=HumanDetector(name='vitdet', device=device),
    human_segmentor=None,
    fov_estimator=None,
)
print('  SAM3D Body ready.')


def merge_with_vitdet(sam3_boxes, vitdet_boxes, iou_thresh=0.4):
    """
    Return SAM3 boxes plus any ViTDet box not already covered by a SAM3 box.

    SAM3 boxes take priority; ViTDet only fills gaps (IoU < iou_thresh with all SAM3 boxes).
    """
    if len(vitdet_boxes) == 0:
        return sam3_boxes
    if len(sam3_boxes) == 0:
        return vitdet_boxes
    ix1   = np.maximum(vitdet_boxes[:, 0:1], sam3_boxes[:, 0])
    iy1   = np.maximum(vitdet_boxes[:, 1:2], sam3_boxes[:, 1])
    ix2   = np.minimum(vitdet_boxes[:, 2:3], sam3_boxes[:, 2])
    iy2   = np.minimum(vitdet_boxes[:, 3:4], sam3_boxes[:, 3])
    inter = np.maximum(0.0, ix2 - ix1) * np.maximum(0.0, iy2 - iy1)
    area_v = ((vitdet_boxes[:, 2] - vitdet_boxes[:, 0]) *
              (vitdet_boxes[:, 3] - vitdet_boxes[:, 1]))[:, None]
    area_s = ((sam3_boxes[:, 2] - sam3_boxes[:, 0]) *
              (sam3_boxes[:, 3] - sam3_boxes[:, 1]))[None, :]
    iou   = inter / (area_v + area_s - inter + 1e-8)   # (K_vitdet, M_sam3)
    extra = vitdet_boxes[iou.max(axis=1) < iou_thresh]
    if len(extra) == 0:
        return sam3_boxes
    return np.vstack([sam3_boxes, extra])


def run_sam3d_body(estimator, img_path, img_bgr, K, sam3_ped_dets,
                   bbox_thresh=0.8, iou_merge_thresh=0.4):
    """
    Run SAM3D Body on one image.

    SAM3 pedestrian masks → pixel boxes (primary source).
    ViTDet is run separately and merged in: any ViTDet box with IoU < iou_merge_thresh
    against all SAM3 boxes is kept as a supplementary detection.
    """
    K_t = torch.tensor(K, dtype=torch.float32).unsqueeze(0)

    # SAM3 masks → pixel-space xyxy boxes
    sam3_boxes = []
    for d in sam3_ped_dets:
        ys, xs = np.where(d['binary_mask'])
        if len(xs) == 0:
            continue
        sam3_boxes.append([xs.min(), ys.min(), xs.max(), ys.max()])
    sam3_boxes = (np.array(sam3_boxes, dtype=np.float32)
                  if sam3_boxes else np.empty((0, 4), dtype=np.float32))

    # Run ViTDet separately and merge
    vitdet_boxes = estimator.detector.run_human_detection(
        img_bgr, bbox_thr=bbox_thresh, nms_thr=0.3, default_to_full_image=False
    )
    vitdet_boxes = np.asarray(vitdet_boxes, dtype=np.float32).reshape(-1, 4)
    merged_boxes = merge_with_vitdet(sam3_boxes, vitdet_boxes, iou_thresh=iou_merge_thresh)
    print(f'    SAM3: {len(sam3_boxes)}  ViTDet: {len(vitdet_boxes)}  merged: {len(merged_boxes)}')

    if len(merged_boxes) == 0:
        return []

    with torch.autocast('cuda', enabled=False):
        outputs = estimator.process_one_image(
            img_path, bboxes=merged_boxes, bbox_thr=bbox_thresh, use_mask=False, cam_int=K_t,
        )

    faces = estimator.faces
    if hasattr(faces, 'cpu'):
        faces = faces.detach().cpu().numpy()
    faces = faces.astype(np.int32)

    results = []
    for j, o in enumerate(outputs):
        verts  = np.asarray(o['pred_vertices'],    dtype=np.float32)
        joints = np.asarray(o['pred_keypoints_3d'], dtype=np.float32)
        cam_t  = np.asarray(o['pred_cam_t'],       dtype=np.float32)
        results.append({
            'vertices':  verts + cam_t[None, :],
            'faces':     faces,
            'joints_3d': joints,
            'cam_t':     cam_t,
            'bbox':      merged_boxes[j],   # xyxy, used for SAM3 mask matching in B1
        })
    return results


print('Running SAM3D Body (pedestrians)...')
ped_dets     = sam3_results.get('pedestrian', [])
body_results = run_sam3d_body(
    body_estimator, img_path, img_bgr, K, ped_dets,
    bbox_thresh=BODY_BBOX_THRESH, iou_merge_thresh=SAM3_IOU_MERGE_THRESH,
)
print(f'  {len(body_results)} pedestrian(s) reconstructed.')

# Unload SAM3D Body
del body_estimator, _body_model
gc.collect()
torch.cuda.empty_cache()
print(f'SAM3D Body unloaded. GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')


## 6. B1 — LiDAR tz Correction

In [ ]:
# ── B1: LiDAR tz correction ──────────────────────────────────────────────────
# After SAM3D Body inference, override tz with LiDAR median depth of the
# HDBSCAN-filtered in-mask surface points for each pedestrian.
#
# Why:
#   SAM3D Body computes tz = 2 * focal_length / (bbox_size * s) where s is a
#   scale parameter predicted from image appearance. A small distant person and
#   a large close person look identical in a crop — tz is the main error source.
#
# Aggregation note:
#   HDBSCAN + median path uses the full aggregated cloud (pts_ego_vis / Z_vis).
#   15th-percentile fallback uses anchor-only points (Z_vis_anchor): non-anchor
#   sweep points of a moving pedestrian can be displaced ~0.1-0.3 m, which
#   biases the front-surface percentile estimate.
#
# Mask source per body result:
#   - If a SAM3 mask bbox has IoU > 0.3 with the body's bbox -> use SAM3 binary mask
#     (precise pixel-accurate mask, better for HDBSCAN).
#   - Otherwise (ViTDet-only detection, no SAM3 mask) -> synthesize a rectangular
#     boolean mask from the bbox. ViTDet detections are still corrected this way.
#
# How tx, ty are recomputed:
#   tx = (u_center - cx) / fx * tz_lidar
#   ty = (v_center - cy) / fy * tz_lidar
#   where (u_center, v_center) is the mask/bbox centroid in pixel space.
#
# joints_3d are body-relative (NOT in camera space).
# The reprojection formula is: joints_cam = joints_3d + cam_t
# So updating cam_t alone is sufficient — do NOT shift joints_3d by delta,
# or the correction would be applied twice when projecting.

import sys as _sys
_sys.path.insert(0, str(DEV_ROOT / 'AutoLabeling' / 'src'))
import hdbscan as _hdbscan_b1


def _best_sam3_mask(bbox, sam3_ped_dets, H, W, iou_thresh=0.3):
    x1, y1, x2, y2 = bbox
    best_mask, best_iou = None, iou_thresh
    for d in sam3_ped_dets:
        ys, xs = np.where(d['binary_mask'])
        if len(xs) == 0:
            continue
        mx1, my1, mx2, my2 = float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())
        ix1 = max(x1, mx1); iy1 = max(y1, my1)
        ix2 = min(x2, mx2); iy2 = min(y2, my2)
        inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
        area_b = (x2 - x1) * (y2 - y1)
        area_m = (mx2 - mx1) * (my2 - my1)
        iou = inter / (area_b + area_m - inter + 1e-8)
        if iou > best_iou:
            best_iou, best_mask = iou, d['binary_mask']
    return best_mask


def apply_b1_lidar_correction(body_results, sam3_ped_dets, K,
                               pts_ego_vis, u_vis, v_vis, Z_vis,
                               pts_ego_vis_anchor=None, u_vis_anchor=None,
                               v_vis_anchor=None, Z_vis_anchor=None,
                               min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
                               min_samples=HDBSCAN_MIN_SAMPLES,
                               cluster_eps=HDBSCAN_CLUSTER_EPS,
                               proximity_min_pts=PROXIMITY_MIN_PTS,
                               proximity_ratio=PROXIMITY_RATIO):
    # pts_ego_vis / u_vis / v_vis / Z_vis   -- full aggregated cloud (all sweeps).
    #     Used for HDBSCAN clustering + median depth (main path).
    #
    # pts_ego_vis_anchor / ... / Z_vis_anchor  -- anchor-only cloud.
    #     Used for 15th-percentile fallback when HDBSCAN finds no cluster.
    #     Avoids bias from displaced non-anchor sweep points of moving pedestrians.
    #     Falls back to aggregated Z_vis if anchor-only yields no in-mask points.
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
    H_img, W_img = img_rgb.shape[:2]

    # Integer pixel indices for aggregated cloud
    u_int = np.round(u_vis).astype(int).clip(0, W_img - 1)
    v_int = np.round(v_vis).astype(int).clip(0, H_img - 1)

    # Integer pixel indices for anchor-only cloud (used in fallback)
    if u_vis_anchor is not None:
        u_int_anc = np.round(u_vis_anchor).astype(int).clip(0, W_img - 1)
        v_int_anc = np.round(v_vis_anchor).astype(int).clip(0, H_img - 1)
    else:
        u_int_anc = v_int_anc = None

    for i, r in enumerate(body_results):
        bbox = r.get('bbox')

        # Try to find a matching SAM3 mask via bbox IoU
        mask = _best_sam3_mask(bbox, sam3_ped_dets, H_img, W_img) if bbox is not None else None

        if mask is not None:
            mask_source = 'sam3_mask'
        else:
            # ViTDet-only: synthesize rectangular mask from bbox
            mask = np.zeros((H_img, W_img), dtype=bool)
            if bbox is not None:
                x1, y1, x2, y2 = [int(v) for v in bbox]
                mask[max(0,y1):min(H_img,y2+1), max(0,x1):min(W_img,x2+1)] = True
                mask_source = 'bbox_rect'
            else:
                print(f'  [B1] pedestrian {i}: no bbox available, skipping.')
                r['b1_mode']    = 'no_bbox'
                r['_dbg_lidar'] = None
                continue

        # Find in-mask LiDAR points (aggregated cloud)
        in_mask    = mask[v_int, u_int]
        pts_inmask = pts_ego_vis[in_mask]   # (N, 3) ego space
        Z_inmask   = Z_vis[in_mask]

        if len(pts_inmask) == 0:
            print(f'  [B1] pedestrian {i}: no LiDAR in mask ({mask_source}), skipping.')
            r['b1_mode']    = 'no_lidar'
            r['_dbg_lidar'] = {'pts_ego': np.empty((0, 3), dtype=np.float32),
                               'tz_lidar': float(r['cam_t'][2]),
                               'tz_pred':  float(r['cam_t'][2]),
                               'mode': 'no_lidar', 'mask_source': mask_source}
            continue

        # HDBSCAN filtering — proximity-aware cluster selection
        if len(pts_inmask) < min_cluster_size:
            keep = None
        else:
            _clusterer = _hdbscan_b1.HDBSCAN(
                min_cluster_size=min_cluster_size,
                min_samples=min_samples,
                metric='euclidean',
                cluster_selection_epsilon=cluster_eps,
            )
            _labels_b1 = _clusterer.fit_predict(pts_inmask)
            _unique_b1, _counts_b1 = np.unique(_labels_b1[_labels_b1 >= 0], return_counts=True)
            if len(_unique_b1) == 0:
                keep = None
            else:
                if USE_OLD_HDBSCAN:
                    # Old: always pick the largest cluster — no proximity selection
                    _dominant_b1 = _unique_b1[np.argmax(_counts_b1)]
                else:
                    # New: proximity-aware — prefer closer cluster when two are roughly equal
                    _order_b1   = np.argsort(_counts_b1)[::-1]
                    _biggest_b1 = int(_counts_b1[_order_b1[0]])
                    if (_biggest_b1 >= proximity_min_pts
                            and len(_order_b1) >= 2
                            and _counts_b1[_order_b1[1]] >= proximity_ratio * _biggest_b1):
                        _c0b = _unique_b1[_order_b1[0]]
                        _c1b = _unique_b1[_order_b1[1]]
                        _d0b = float(np.linalg.norm(pts_inmask[_labels_b1 == _c0b].mean(axis=0)))
                        _d1b = float(np.linalg.norm(pts_inmask[_labels_b1 == _c1b].mean(axis=0)))
                        _dominant_b1 = _c0b if _d0b <= _d1b else _c1b
                    else:
                        _dominant_b1 = _unique_b1[_order_b1[0]]
                keep = _labels_b1 == _dominant_b1
        tz_pred = float(r['cam_t'][2])

        if keep is None:
            # HDBSCAN found no coherent surface cluster.
            # Use anchor-only in-mask depths for the 15th-percentile fallback:
            # non-anchor sweep points of a moving pedestrian are displaced
            # (~0.1-0.3 m), which can bias the front-surface estimate.
            if u_int_anc is not None:
                in_mask_anc  = mask[v_int_anc, u_int_anc]
                Z_inmask_anc = Z_vis_anchor[in_mask_anc]
            else:
                Z_inmask_anc = np.array([], dtype=np.float32)

            # Fall back to aggregated if anchor has no in-mask points at all
            Z_for_pct  = Z_inmask_anc if len(Z_inmask_anc) > 0 else Z_inmask
            pct_source = 'anchor' if len(Z_inmask_anc) > 0 else 'aggregated'

            tz_lidar  = float(np.percentile(Z_for_pct, 15))
            ys, xs    = np.where(mask)
            u_cen     = float(xs.mean())
            v_cen     = float(ys.mean())
            tx_new    = (u_cen - cx) / fx * tz_lidar
            ty_new    = (v_cen - cy) / fy * tz_lidar
            cam_t_new = np.array([tx_new, ty_new, tz_lidar], dtype=np.float32)
            delta     = cam_t_new - r['cam_t']
            r['vertices'] = r['vertices'] + delta[None, :]
            r['cam_t']    = cam_t_new
            b1_mode       = f'{mask_source}+p15_{pct_source}_fallback'
            r['b1_mode']  = b1_mode
            r['_dbg_lidar'] = {'pts_ego': pts_inmask.copy(),
                               'tz_lidar': tz_lidar, 'tz_pred': tz_pred,
                               'mode': b1_mode, 'mask_source': mask_source}
            print(f'  [B1] pedestrian {i}: HDBSCAN no cluster ({len(pts_inmask)} agg pts, '
                  f'{len(Z_inmask_anc)} anchor pts), '
                  f'p15/{pct_source} tz {tz_pred:.2f} -> {tz_lidar:.2f} m '
                  f'(delta={tz_lidar - tz_pred:+.2f} m)')
            continue
        else:
            Z_clean  = Z_inmask[keep]
            b1_mode  = f'{mask_source}+hdbscan'

        tz_lidar = float(np.median(Z_clean))

        # Recompute tx, ty from mask centroid and new tz
        ys, xs = np.where(mask)
        u_cen  = float(xs.mean())
        v_cen  = float(ys.mean())
        tx_new = (u_cen - cx) / fx * tz_lidar
        ty_new = (v_cen - cy) / fy * tz_lidar

        cam_t_new = np.array([tx_new, ty_new, tz_lidar], dtype=np.float32)
        delta     = cam_t_new - r['cam_t']

        r['vertices'] = r['vertices'] + delta[None, :]
        r['cam_t']    = cam_t_new
        r['b1_mode']  = b1_mode
        r['_dbg_lidar'] = {'pts_ego': pts_inmask.copy(),
                           'tz_lidar': tz_lidar, 'tz_pred': tz_pred,
                           'mode': b1_mode, 'mask_source': mask_source}

        print(f'  [B1] pedestrian {i}: tz {tz_pred:.2f} -> {tz_lidar:.2f} m  '
              f'(delta={tz_lidar - tz_pred:+.2f} m)  [{b1_mode}, {len(Z_clean)} pts]')

    return body_results


if B1_LIDAR_CORRECTION:
    print('Applying B1 LiDAR tz correction...')
    ped_dets = sam3_results.get('pedestrian', [])
    body_results = apply_b1_lidar_correction(
        body_results, ped_dets, K,
        pts_ego_vis, u_vis, v_vis, Z_vis,
        pts_ego_vis_anchor, u_vis_anchor, v_vis_anchor, Z_vis_anchor,
    )
    print('B1 correction done.')
else:
    print('B1_LIDAR_CORRECTION=False — using baseline SAM3D Body tz.')
    for r in body_results:
        r['b1_mode']    = 'baseline'
        r['_dbg_lidar'] = None

In [ ]:
from autolabeling.utils.lidar import filter_inmask_lidar_hdbscan

# ── BEV: HDBSCAN filtering quality — all pedestrians in one plot (plotly) ─────
# Gray  = all visible LiDAR points (camera FOV, subsampled)
# Color = dominant cluster kept by filter_inmask_lidar_hdbscan (one color per ped)
# Points not in the dominant cluster (noise + smaller clusters) are not shown.

PED_PALETTE = [
    'rgb(230,25,75)',  'rgb(60,180,75)',   'rgb(67,99,216)',  'rgb(245,130,49)',
    'rgb(145,30,180)', 'rgb(66,212,244)',  'rgb(240,50,230)', 'rgb(191,239,69)',
    'rgb(250,190,212)','rgb(70,153,144)',  'rgb(220,190,255)','rgb(154,99,36)',
]

traces_hdb = []

# Background: all visible LiDAR (subsampled)
_step = max(1, len(pts_ego_vis) // 6000)
_bg   = pts_ego_vis[::_step]
traces_hdb.append(go.Scatter(
    x=_bg[:, 0], y=_bg[:, 1],
    mode='markers',
    marker=dict(size=1.5, color='rgba(100,100,100,0.35)'),
    name='all visible LiDAR', hoverinfo='skip',
))

for i, r in enumerate(body_results):
    dbg = r.get('_dbg_lidar')
    if dbg is None or len(dbg['pts_ego']) == 0:
        continue

    pts = dbg['pts_ego']          # (N, 3) ego: X=forward, Y=left
    col = PED_PALETTE[i % len(PED_PALETTE)]

    keep = filter_inmask_lidar_hdbscan(pts, min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, min_samples=HDBSCAN_MIN_SAMPLES, cluster_eps=HDBSCAN_CLUSTER_EPS)
    if keep is None:
        continue

    tz_l = dbg['tz_lidar']
    tz_p = dbg['tz_pred']
    mode = dbg.get('mode', '?')
    traces_hdb.append(go.Scatter(
        x=pts[keep, 0], y=pts[keep, 1],
        mode='markers',
        marker=dict(size=6, color=col, opacity=0.95),
        name=f'ped {i}  tz {tz_p:.1f}\u2192{tz_l:.1f} m  [{keep.sum()} pts  {mode}]',
    ))

# Ego vehicle
traces_hdb.append(go.Scatter(
    x=[0], y=[0], mode='markers',
    marker=dict(symbol='arrow-up', angle=90, size=16, color='gold',
                line=dict(color='black', width=1)),
    name='ego',
))

mode_str = 'B1' if B1_LIDAR_CORRECTION else 'Baseline'
fig_hdb = go.Figure(data=traces_hdb)
fig_hdb.update_layout(
    title=f'HDBSCAN dominant cluster \u2014 BEV, ego frame  |  {mode_str}  |  '
          f'scene {SCENE_IDX} frame {FRAME_IDX}  '
          f'(min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}, min_samples={HDBSCAN_MIN_SAMPLES}, ε={HDBSCAN_CLUSTER_EPS} m)',
    xaxis=dict(title='X ego (forward, m)', scaleanchor='y', scaleratio=1,
               showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False),
    yaxis=dict(title='Y ego (left, m)',
               showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False),
    plot_bgcolor='rgb(15,15,25)', paper_bgcolor='rgb(15,15,25)',
    font=dict(color='white'),
    legend=dict(bgcolor='rgba(0,0,0,0.55)', font=dict(size=10)),
    width=1100, height=560, margin=dict(l=0, r=0, t=45, b=0),
)
fig_hdb.show()


## 6.5 B1 Stage 2 — Ground Anchoring (PseudoLabeler)

After Stage 1 sets `tz` (and recomputes `tx`, `ty` from the mask centroid), pedestrian
meshes can still float: `cam_t` is placed at the torso/pelvis depth, not the foot level.
Stage 2 anchors each mesh so its lowest vertices sit on the local road surface.

**PseudoLabeler** fits a small MLP `gθ: R² → R` on the raw LiDAR sweep using an
asymmetric loss, producing a continuous, queryable ground surface. Since the MLP is
defined over the full 2D plane, it interpolates smoothly between sparse LiDAR returns
and can be evaluated at any pedestrian `(x, y)` — even at range where LiDAR rings spread out.

Because Stage 1 already corrected `tx` and `ty`, the `(x, y)` position used to query
the ground surface is already in the right place.

**Anchoring step:**
1. Query `gθ(ped_xy)` → `z_ground`
2. Convert mesh vertices to ego space → `z_foot = min vertex Z`
3. `z_shift = z_ground − z_foot`
4. Convert the ego Z shift back to camera space via `R_c2e.T`
5. Apply to `vertices` and `cam_t`. OBB is refit in the next cell.
6. `joints_3d` are body-relative and are **NOT** shifted.

**Fallback:** global RANSAC plane if `pseudolabeler_model` is None (fitting failed).


In [ ]:
from sklearn.linear_model import RANSACRegressor

_RANSAC_Z_MIN    = -1.0
_RANSAC_Z_MAX    =  0.5
_RANSAC_RESIDUAL =  0.10


def _query_pseudolabeler(ped_xy, pl_model, device_pl):
    """Query ground height at ped_xy using the fitted PseudoLabeler MLP."""
    xy_t = torch.tensor([[ped_xy[0], ped_xy[1]]], dtype=torch.float32).to(device_pl)
    with torch.no_grad():
        z_ground = float(pl_model(xy_t).item())
    return z_ground, 'pseudolabeler'


def _ransac_fallback(ped_xy, pts_ego_f64):
    """Global RANSAC ground plane — used only when PseudoLabeler is unavailable."""
    g_cands = pts_ego_f64[(pts_ego_f64[:, 2] >= _RANSAC_Z_MIN) &
                          (pts_ego_f64[:, 2] <= _RANSAC_Z_MAX)]
    ransac  = RANSACRegressor(min_samples=3, residual_threshold=_RANSAC_RESIDUAL,
                              max_trials=500, random_state=42)
    ransac.fit(g_cands[:, :2], g_cands[:, 2])
    return float(ransac.predict([ped_xy])[0]), f'ransac_global ({len(g_cands)} pts)'


def apply_b2_ground_anchoring(body_results, pts_ego, R_c2e, t_c2e,
                               pl_model=None, device_pl=None):
    """
    Shift each pedestrian mesh so its lowest vertex aligns with the local ground surface.

    Primary path  (pl_model not None): query fitted PseudoLabeler MLP at pedestrian (x,y).
    Fallback      (pl_model is None):  global RANSAC plane.

    Only vertices and cam_t are shifted.  joints_3d are body-relative — NOT shifted.
    OBB must be refit after this step.
    """
    pts_ego_f64 = pts_ego.astype(np.float64)
    path_label  = 'PseudoLabeler MLP' if pl_model is not None else 'RANSAC fallback'
    print(f'  B2 path: {path_label}')

    for i, r in enumerate(body_results):
        cam_t_ego = R_c2e @ r['cam_t'].astype(np.float64) + t_c2e
        ped_xy    = cam_t_ego[:2]

        try:
            if pl_model is not None:
                z_ground, method = _query_pseudolabeler(ped_xy, pl_model, device_pl)
            else:
                z_ground, method = _ransac_fallback(ped_xy, pts_ego_f64)
        except Exception as e:
            print(f'  [B2] ped {i}: query failed ({e}), using RANSAC fallback.')
            z_ground, method = _ransac_fallback(ped_xy, pts_ego_f64)

        verts_ego = (R_c2e @ r['vertices'].astype(np.float64).T).T + t_c2e
        z_foot    = float(verts_ego[:, 2].min())
        z_shift   = z_ground - z_foot

        delta_cam     = (R_c2e.T @ np.array([0.0, 0.0, z_shift])).astype(np.float32)
        r['vertices'] = r['vertices'] + delta_cam[None, :]
        r['cam_t']    = r['cam_t']    + delta_cam
        r['b2_mode']  = method
        r['_b2_debug'] = dict(ped_xy=ped_xy, z_ground=z_ground, z_foot=z_foot,
                              z_shift=z_shift, method=method)

        print(f'  [B2] ped {i}: foot={z_foot:.2f} m  ground={z_ground:.2f} m  '
              f'shift={z_shift:+.2f} m  [{method}]')

    return body_results


if B2_GROUND_ANCHORING:
    print('Applying B2 ground anchoring...')
    body_results = apply_b2_ground_anchoring(
        body_results, pts_ego, R_c2e, t_c2e,
        pl_model=pseudolabeler_model,
        device_pl=_device_pl,
    )
    print('B2 done.')
else:
    print('B2_GROUND_ANCHORING=False — skipping ground anchoring.')
    for r in body_results:
        r['b2_mode']   = 'skipped'
        r['_b2_debug'] = None

In [ ]:
# ── B2 ground estimation: intermediate visualization ──────────────────────────
# Left : BEV — PseudoLabeler ground surface sampled on a grid (coloured by Z),
#         overlaid with pedestrian positions and estimated ground Z.
# Right: per-pedestrian bar chart — foot Z before B2, ground Z, and shift.

import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(18, 7), facecolor='#111111')
gs  = gridspec.GridSpec(1, 2, width_ratios=[2.5, 1], figure=fig)

# ── Left: BEV ────────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0])
ax.set_facecolor('#111111')

if pseudolabeler_model is not None and B2_GROUND_ANCHORING:
    # Sample PseudoLabeler surface on a grid covering the LiDAR footprint
    _x_min, _x_max = float(pts_ego[:, 0].min()), float(pts_ego[:, 0].max())
    _y_min, _y_max = float(pts_ego[:, 1].min()), float(pts_ego[:, 1].max())
    _res   = 0.5   # grid resolution [m]
    _gx    = np.arange(_x_min, _x_max, _res)
    _gy    = np.arange(_y_min, _y_max, _res)
    _GX, _GY = np.meshgrid(_gx, _gy)
    _grid_xy = np.stack([_GX.ravel(), _GY.ravel()], axis=1)

    with torch.no_grad():
        _grid_t = torch.tensor(_grid_xy, dtype=torch.float32).to(_device_pl)
        _grid_z = pseudolabeler_model(_grid_t).cpu().numpy()

    sc = ax.scatter(_grid_xy[:, 0], _grid_xy[:, 1],
                    c=_grid_z, cmap='RdYlGn', s=4, alpha=0.6,
                    vmin=np.percentile(_grid_z, 2), vmax=np.percentile(_grid_z, 98),
                    zorder=1, rasterized=True)
    plt.colorbar(sc, ax=ax, label='Z ego (m)', fraction=0.02, pad=0.01)
    title_src = f'PseudoLabeler ground surface ({_res} m grid)'
else:
    # Fallback: show raw LiDAR in the RANSAC Z band
    _band = pts_ego[(pts_ego[:, 2] >= _RANSAC_Z_MIN) & (pts_ego[:, 2] <= _RANSAC_Z_MAX)]
    _step = max(1, len(_band) // 8000)
    sc    = ax.scatter(_band[::_step, 0], _band[::_step, 1],
                       c=_band[::_step, 2], cmap='RdYlGn', s=1, alpha=0.5,
                       zorder=1, rasterized=True)
    plt.colorbar(sc, ax=ax, label='Z ego (m)', fraction=0.02, pad=0.01)
    title_src = 'RANSAC fallback (raw LiDAR Z band)'

PED_PALETTE_B2 = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4',
                  '#42d4f4','#f032e6','#bfef45','#fabed4','#469990']

for i, r in enumerate(body_results):
    dbg = r.get('_b2_debug')
    if dbg is None:
        continue
    col = PED_PALETTE_B2[i % len(PED_PALETTE_B2)]
    xy  = dbg['ped_xy']
    ax.scatter(xy[0], xy[1], s=120, c=col, marker='*', zorder=5,
               edgecolors='white', linewidths=0.5)
    ax.annotate(f'ped {i}\ng={dbg["z_ground"]:.2f} m\nshift={dbg["z_shift"]:+.2f} m',
                (xy[0], xy[1]), textcoords='offset points', xytext=(8, 4),
                color=col, fontsize=7.5,
                bbox=dict(boxstyle='round,pad=0.2', fc='#111111', alpha=0.7, ec='none'))

ax.scatter(0, 0, s=120, c='gold', marker='D', zorder=6)
ax.set_aspect('equal')
ax.set_xlabel('X ego (forward, m)', color='white')
ax.set_ylabel('Y ego (left, m)', color='white')
ax.tick_params(colors='white')
ax.set_title(f'B2 ground estimation — {title_src}', color='white', fontsize=10)
ax.grid(True, alpha=0.15, color='white')

# ── Right: per-pedestrian bar chart ──────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor('#1a1a2e')

n_peds    = len(body_results)
xs        = np.arange(n_peds)
w         = 0.3
z_foots   = [r['_b2_debug']['z_foot']   if r.get('_b2_debug') else 0 for r in body_results]
z_grounds = [r['_b2_debug']['z_ground'] if r.get('_b2_debug') else 0 for r in body_results]
z_shifts  = [r['_b2_debug']['z_shift']  if r.get('_b2_debug') else 0 for r in body_results]

ax2.bar(xs - w, z_foots,   width=w, label='foot Z before B2', color='#4a90d9', alpha=0.85)
ax2.bar(xs,     z_grounds, width=w, label='ground Z (PseudoLabeler)', color='#50c878', alpha=0.85)
ax2.bar(xs + w, z_shifts,  width=w, label='shift applied', color='#ff6b6b', alpha=0.85)
ax2.axhline(0, color='white', linewidth=0.6, alpha=0.5)
ax2.set_xticks(xs)
ax2.set_xticklabels([f'ped {i}' for i in range(n_peds)], color='white')
ax2.set_ylabel('Z ego (m)', color='white')
ax2.tick_params(colors='white')
ax2.set_title('B2 shift per pedestrian', color='white', fontsize=10)
ax2.legend(fontsize=8, facecolor='#222222', labelcolor='white', framealpha=0.85)
for sp in ax2.spines.values():
    sp.set_color('gray')

fig.patch.set_facecolor('#111111')
plt.tight_layout()
plt.show()


## 7. Orientation Estimation

In [ ]:
# ── Orientation estimation ────────────────────────────────────────────────────

_NOSE       = 0
_L_SHOULDER = 5
_R_SHOULDER = 6


def facing_direction(joints):
    """
    Estimate horizontal facing direction of a pedestrian from MHR70 skeleton.

    Parameters
    ----------
    joints : (70, 3) float32  MHR70 keypoints in body-relative space

    Returns
    -------
    fwd : (3,) float32  unit vector in XZ plane, Y=0. Fully resolved (no 180° ambiguity).
    """
    shoulder_vec = joints[_R_SHOULDER] - joints[_L_SHOULDER]
    sv_xz = np.array([shoulder_vec[0], 0.0, shoulder_vec[2]], dtype=np.float32)
    sv_xz /= np.linalg.norm(sv_xz) + 1e-8

    # 90° rotation in XZ gives a facing candidate
    fwd = np.array([-sv_xz[2], 0.0, sv_xz[0]], dtype=np.float32)

    # Disambiguate via nose: nose should be in front of the shoulder midpoint
    shoulder_mid = (joints[_L_SHOULDER] + joints[_R_SHOULDER]) / 2.0
    if np.dot(fwd, joints[_NOSE] - shoulder_mid) < 0:
        fwd = -fwd

    return fwd


def estimate_orientation(result, kind):
    """
    Estimate orientation for a single detection.

    Parameters
    ----------
    result : dict from body_results
    kind   : 'pedestrian'

    Returns
    -------
    fwd : (3,) unit vector in camera XZ plane, or None (ambiguous)
    """
    if kind == 'pedestrian':
        return facing_direction(result['joints_3d'])
    return None


# Attach orientation to results
for r in body_results:
    r['orientation_fwd'] = estimate_orientation(r, 'pedestrian')

print('Orientation estimation done.')
for i, r in enumerate(body_results):
    print(f'  Pedestrian {i}: fwd = {r["orientation_fwd"].round(3)}')


## 8. OBB Fitting

In [ ]:
# ── OBB Fitting ─────────────────────────────────────────────────────────────────

_BBOX_EDGES = [
    (0,1),(1,2),(2,3),(3,0),   # bottom face
    (4,5),(5,6),(6,7),(7,4),   # top face
    (0,4),(1,5),(2,6),(3,7),   # verticals
]


def compute_obb_pedestrian(vertices, fwd):
    """
    Fit an OBB aligned to the pedestrian's facing direction.

    Parameters
    ----------
    vertices : (V, 3) float32  mesh vertices in camera space (R3)
    fwd      : (3,)   float32  unit facing vector in XZ plane (Y=0)

    Returns
    -------
    corners : (8, 3) float32  OBB corners in camera space
    center  : (3,)   float32
    dims    : (3,)   float32  [width, height, depth]
    """
    right = np.array([fwd[2], 0.0, -fwd[0]], dtype=np.float32)   # 90° CW from fwd in XZ
    up    = np.array([0.0, 1.0,  0.0], dtype=np.float32)

    r_proj = vertices @ right
    y_proj = vertices[:, 1]
    f_proj = vertices @ fwd

    r_min, r_max = float(r_proj.min()), float(r_proj.max())
    y_min, y_max = float(y_proj.min()), float(y_proj.max())
    f_min, f_max = float(f_proj.min()), float(f_proj.max())

    corners = np.array([
        r_min*right + y_min*up + f_min*fwd,
        r_max*right + y_min*up + f_min*fwd,
        r_max*right + y_max*up + f_min*fwd,
        r_min*right + y_max*up + f_min*fwd,
        r_min*right + y_min*up + f_max*fwd,
        r_max*right + y_min*up + f_max*fwd,
        r_max*right + y_max*up + f_max*fwd,
        r_min*right + y_max*up + f_max*fwd,
    ], dtype=np.float32)

    center = corners.mean(axis=0)
    dims   = np.array([r_max-r_min, y_max-y_min, f_max-f_min], dtype=np.float32)
    return corners, center, dims


def fit_obb(result, kind, R_c2e, t_c2e):
    """
    Fit an OBB for a single detection.

    Parameters
    ----------
    result : dict
    kind   : 'pedestrian'
    R_c2e  : (3, 3)  camera → ego rotation
    t_c2e  : (3,)    camera → ego translation

    Returns
    -------
    Adds 'obb_corners', 'obb_center', 'obb_dims' to result in-place.
    """
    verts = result['vertices']
    if kind == 'pedestrian':
        fwd = result['orientation_fwd']
        corners, center, dims = compute_obb_pedestrian(verts, fwd)
        result['obb_corners'] = corners    # camera space
        result['obb_center']  = center
        result['obb_dims']    = dims
        result['obb_yaw']     = float(np.arctan2(fwd[0], fwd[2]))  # approx yaw in camera


# Fit OBBs
for r in body_results:
    fit_obb(r, 'pedestrian', R_c2e, t_c2e)

print('OBB fitting done.')
for i, r in enumerate(body_results):
    print(f'  Pedestrian {i}: dims (W×H×D) = {r["obb_dims"].round(3)} m (camera-relative)')


## 9. Skeleton + Mesh Reprojection (2D)

In [ ]:
# ── Skeleton + Mesh reprojection onto image ───────────────────────────────────
# Mesh: rendered via SAM3D Body Renderer (same path as run_sam3d_bodies.py).
# Skeleton: 3D joints (body-relative) projected via joints_3d + cam_t = camera space.
#
# joints_3d are body-relative. B1 does NOT shift them; only cam_t and vertices
# (absolute camera space) are updated. So joints_cam = joints_3d + cam_t is always correct.

from tools.vis_utils import visualizer as _vis
from sam_3d_body.visualization.renderer import Renderer


def project_points(pts_cam, K):
    """(N, 3) camera-space → (N, 2) pixel coords."""
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
    u = pts_cam[:, 0] / pts_cam[:, 2] * fx + cx
    v = pts_cam[:, 1] / pts_cam[:, 2] * fy + cy
    return np.stack([u, v], axis=1)


# ── Mesh overlay (farthest-first so nearer people render on top) ─────────────
canvas_bgr = img_bgr.copy()

if body_results:
    faces    = body_results[0]['faces']
    renderer = Renderer(focal_length=fx, faces=faces)

    for r in sorted(body_results, key=lambda r: float(r['cam_t'][2]), reverse=True):
        # r['vertices'] is in absolute camera space (pred_vertices + cam_t, shifted by B1).
        # Renderer expects body-relative vertices + cam_t separately.
        verts_rel = r['vertices'] - r['cam_t'][None, :]
        rendered  = renderer(
            vertices=verts_rel,
            cam_t=r['cam_t'],
            image=canvas_bgr,
            full_frame=False,
            camera_center=[cx, cy],
        )
        # rendered is RGB [0, 1] → back to BGR uint8 for next iteration
        canvas_bgr = cv2.cvtColor(
            (rendered * 255).clip(0, 255).astype(np.uint8), cv2.COLOR_RGB2BGR
        )

vis_img = cv2.cvtColor(canvas_bgr, cv2.COLOR_BGR2RGB)

# ── Skeleton overlay ──────────────────────────────────────────────────────────
for r in sorted(body_results, key=lambda r: float(r['cam_t'][2]), reverse=True):
    joints_cam = r['joints_3d'] + r['cam_t'][None, :]
    kp2d       = project_points(joints_cam, K)
    kp2d_vis   = np.concatenate([kp2d, np.ones((len(kp2d), 1))], axis=-1)
    vis_img    = _vis.draw_skeleton(vis_img, kp2d_vis)

fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(vis_img)
ax.axis('off')
mode_str = 'B1 (LiDAR tz corrected)' if B1_LIDAR_CORRECTION else 'Baseline'
ax.set_title(f'Skeleton + Mesh reprojection — {mode_str}')
plt.tight_layout()
plt.show()


In [ ]:
# ── View frustum constants (tweak here) ──────────────────────────────────────
VIZ3D_RADIUS = 55.0   # m — max XY distance from ego (sides + front)
VIZ3D_X_MIN  = -5.0   # m — clip rear; keep points no more than 5 m behind ego


def cam_to_ego(pts, R_c2e, t_c2e):
    """Transform (N, 3) camera-space (R3) points to ego frame."""
    return (R_c2e @ pts.T).T + t_c2e


BODY_COLORS = [
    'rgb(255,100,100)', 'rgb(255,165,0)', 'rgb(255,220,0)',
    'rgb(200,100,255)', 'rgb(100,200,255)',
]

_BBOX_EDGES = [
    (0,1),(1,2),(2,3),(3,0),
    (4,5),(5,6),(6,7),(7,4),
    (0,4),(1,5),(2,6),(3,7),
]

_SKEL_EDGES = [
    (0, 1), (0, 2),
    (5, 6),
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (5, 11), (6, 12),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
]


def _obb_lines(corners, color):
    xs, ys, zs = [], [], []
    for e0, e1 in _BBOX_EDGES:
        xs += [corners[e0, 0], corners[e1, 0], None]
        ys += [corners[e0, 1], corners[e1, 1], None]
        zs += [corners[e0, 2], corners[e1, 2], None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                        line=dict(color=color, width=3), showlegend=False, hoverinfo='skip')


def _skel_lines(joints_ego, color):
    xs, ys, zs = [], [], []
    for e0, e1 in _SKEL_EDGES:
        if e0 < len(joints_ego) and e1 < len(joints_ego):
            xs += [joints_ego[e0, 0], joints_ego[e1, 0], None]
            ys += [joints_ego[e0, 1], joints_ego[e1, 1], None]
            zs += [joints_ego[e0, 2], joints_ego[e1, 2], None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                        line=dict(color=color, width=4), showlegend=False, hoverinfo='skip')


traces = []

# LiDAR pointcloud — filtered to frustum, subsampled for plotly performance
_pts_all = pts_ego_vis.astype(np.float64)
_r_xy    = np.sqrt(_pts_all[:, 0]**2 + _pts_all[:, 1]**2)
_pts_flt = _pts_all[(_r_xy <= VIZ3D_RADIUS) & (_pts_all[:, 0] >= VIZ3D_X_MIN)]
_step    = max(1, len(_pts_flt) // 8000)
_pts     = _pts_flt[::_step]
traces.append(go.Scatter3d(
    x=_pts[:, 0], y=_pts[:, 1], z=_pts[:, 2],
    mode='markers',
    marker=dict(size=1.5, color=_pts[:, 2], colorscale='Viridis', opacity=0.25,
                colorbar=dict(title='Z ego (m)', thickness=8, len=0.5)),
    name='LiDAR (all)', showlegend=True,
))

# PseudoLabeler ground surface — clipped to same frustum
if pseudolabeler_model is not None and B2_GROUND_ANCHORING:
    _gx  = np.arange(VIZ3D_X_MIN, VIZ3D_RADIUS, 1.0)
    _gy  = np.arange(-VIZ3D_RADIUS, VIZ3D_RADIUS, 1.0)
    _GX, _GY = np.meshgrid(_gx, _gy)
    _gxy = np.stack([_GX.ravel(), _GY.ravel()], axis=1)
    # keep only points within the XY radius
    _gr  = np.sqrt(_gxy[:, 0]**2 + _gxy[:, 1]**2)
    _gxy = _gxy[_gr <= VIZ3D_RADIUS]
    with torch.no_grad():
        _gz = pseudolabeler_model(
            torch.tensor(_gxy, dtype=torch.float32).to(_device_pl)
        ).cpu().numpy()
    traces.append(go.Scatter3d(
        x=_gxy[:, 0], y=_gxy[:, 1], z=_gz,
        mode='markers',
        marker=dict(size=2, color='rgb(80,220,120)', opacity=0.4),
        name='PseudoLabeler ground', showlegend=True,
    ))

# Pedestrian meshes, skeletons, OBBs
for i, r in enumerate(body_results):
    col       = BODY_COLORS[i % len(BODY_COLORS)]
    verts_ego = cam_to_ego(r['vertices'].astype(np.float64), R_c2e, t_c2e)
    f         = r['faces']
    traces.append(go.Mesh3d(
        x=verts_ego[:, 0], y=verts_ego[:, 1], z=verts_ego[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color=col, opacity=0.6,
        name=f'pedestrian_{i}', showlegend=True,
    ))
    joints_cam = r['joints_3d'] + r['cam_t'][None, :]
    joints_ego = cam_to_ego(joints_cam.astype(np.float64), R_c2e, t_c2e)
    traces.append(_skel_lines(joints_ego, col))
    obb_ego = cam_to_ego(r['obb_corners'].astype(np.float64), R_c2e, t_c2e)
    traces.append(_obb_lines(obb_ego, col))

traces.append(go.Scatter3d(
    x=[0], y=[0], z=[0], mode='markers',
    marker=dict(size=6, color='yellow', symbol='diamond'),
    name='ego',
))

mode_str = 'B1+B2' if (B1_LIDAR_CORRECTION and B2_GROUND_ANCHORING) else \
           'B1' if B1_LIDAR_CORRECTION else 'Baseline'
fig = go.Figure(data=traces)
fig.update_layout(
    title=f'SAM3D Body — ego frame  [{mode_str}]',
    scene=dict(
        xaxis_title='X ego (forward, m)',
        yaxis_title='Y ego (left, m)',
        zaxis_title='Z ego (up, m)',
        xaxis=dict(range=[VIZ3D_X_MIN, VIZ3D_RADIUS]),
        yaxis=dict(range=[-VIZ3D_RADIUS, VIZ3D_RADIUS]),
        aspectmode='data',
    ),
    legend=dict(bgcolor='rgba(0,0,0,0.4)', font=dict(color='white')),
    paper_bgcolor='rgb(15,15,25)',
    font=dict(color='white'),
)
fig.show()


In [ ]:
# ── BEV: Final OBB footprints (ego frame) — plotly ───────────────────────────

from nuscenes.utils.data_classes import Box as NuScenesBox

def _rgb_to_rgba(c, a):
    nums = [x.strip() for x in c.replace('rgb(', '').replace(')', '').split(',')]
    return f'rgba({nums[0]},{nums[1]},{nums[2]},{a})'

BODY_COLORS = [
    'rgb(255,100,100)', 'rgb(255,165,0)', 'rgb(255,220,0)',
    'rgb(200,100,255)', 'rgb(100,200,255)',
]
FOOT_IDX = [0, 1, 5, 4]

# ── Load GT annotations for the anchor frame ─────────────────────────────────
# ego_pose at the anchor camera timestamp  →  used for global→ego transform
_cam_ep    = nusc.get('ego_pose', cam_sd['ego_pose_token'])
_R_e2g_anc = Quaternion(_cam_ep['rotation']).rotation_matrix.astype(np.float64)
_t_e2g_anc = np.array(_cam_ep['translation'], dtype=np.float64)

# Find the nearest keyframe (sample) so we always have annotations.
# For true keyframes this is instant; for non-keyframes (e.g. ECP high-rate
# frames) we walk the scene's sample list and pick the closest by timestamp.
if cam_sd['is_key_frame']:
    _sample_token = cam_sd['sample_token']
else:
    _best_tok, _best_dt = None, float('inf')
    _tok = scene['first_sample_token']
    while _tok:
        _s  = nusc.get('sample', _tok)
        _dt = abs(_s['timestamp'] - cam_sd['timestamp'])
        if _dt < _best_dt:
            _best_dt, _best_tok = _dt, _tok
        _tok = _s['next']
    _sample_token = _best_tok

_sample    = nusc.get('sample', _sample_token)
_ann_tokens = _sample['anns']
print(f'GT: {len(_ann_tokens)} box(es)  (sample={_sample_token})')

# Transform each GT box from global frame → anchor ego frame
gt_boxes_ego = []
for _ann_tok in _ann_tokens:
    _ann = nusc.get('sample_annotation', _ann_tok)
    _box = NuScenesBox(
        center=_ann['translation'],
        size=_ann['size'],          # wlh
        orientation=Quaternion(_ann['rotation']),
        name=_ann['category_name'],
    )
    # global → anchor ego: translate then rotate
    _box.translate(-_t_e2g_anc)
    _box.rotate(Quaternion(matrix=_R_e2g_anc).inverse)
    # BEV footprint: corners indices [0,1,5,4] trace the ground-plane rectangle
    # (same ordering as FOOT_IDX used for predicted OBBs)
    _c = _box.corners()   # (3, 8)  in ego frame after the transforms above
    gt_boxes_ego.append({
        'foot_xy': _c[:2, [0, 1, 5, 4]],   # (2, 4)  rows: X, Y
        'name':    _ann['category_name'],
    })

# ── Background: full LiDAR sweep (downsampled, BEV window) ───────────────────
_rng   = np.random.default_rng(42)
_n_bg  = min(30_000, len(pts_ego))
_idx_b = _rng.choice(len(pts_ego), _n_bg, replace=False)
_pts_b = pts_ego[_idx_b]
_bev_m = (np.abs(_pts_b[:, 0]) < 70) & (np.abs(_pts_b[:, 1]) < 35)
_pts_b = _pts_b[_bev_m]

obb_traces = [go.Scatter(
    x=_pts_b[:, 0], y=_pts_b[:, 1],
    mode='markers',
    marker=dict(size=1, color='rgba(100,130,160,0.35)'),
    name='LiDAR (all)', hoverinfo='skip',
)]

obb_annotations = []

# ── Predicted OBB footprints ──────────────────────────────────────────────────
for i, r in enumerate(body_results):
    col      = BODY_COLORS[i % len(BODY_COLORS)]
    col_fill = _rgb_to_rgba(col, 0.20)
    obb_ego  = (R_c2e @ r['obb_corners'].astype(np.float64).T).T + t_c2e
    foot_xy  = obb_ego[FOOT_IDX, :2]

    px = list(foot_xy[:, 0]) + [foot_xy[0, 0]]
    py = list(foot_xy[:, 1]) + [foot_xy[0, 1]]
    obb_traces.append(go.Scatter(
        x=px, y=py, mode='lines',
        fill='toself', fillcolor=col_fill,
        line=dict(color=col, width=2),
        name=f'ped {i}  OBB', legendgroup=f'ped_{i}',
    ))

    cen_ego = obb_ego.mean(axis=0)
    obb_traces.append(go.Scatter(
        x=[cen_ego[0]], y=[cen_ego[1]], mode='text',
        text=[str(i)], textfont=dict(color='white', size=11),
        showlegend=False, hoverinfo='skip', legendgroup=f'ped_{i}',
    ))

    fwd_ego   = R_c2e @ r['orientation_fwd'].astype(np.float64)
    arrow_len = float(max(r['obb_dims'][2] * 0.7, 0.4))
    obb_annotations.append(dict(
        x=cen_ego[0] + fwd_ego[0] * arrow_len,
        y=cen_ego[1] + fwd_ego[1] * arrow_len,
        ax=cen_ego[0], ay=cen_ego[1],
        xref='x', yref='y', axref='x', ayref='y',
        arrowhead=3, arrowsize=1.5, arrowwidth=2,
        arrowcolor=col, showarrow=True, text='',
    ))

# ── GT boxes (white dashed) ───────────────────────────────────────────────────
_gt_legend_shown = False
for _gb in gt_boxes_ego:
    _fx = list(_gb['foot_xy'][0]) + [_gb['foot_xy'][0, 0]]
    _fy = list(_gb['foot_xy'][1]) + [_gb['foot_xy'][1, 0]]
    obb_traces.append(go.Scatter(
        x=_fx, y=_fy, mode='lines',
        line=dict(color='white', width=1, dash='dash'),
        name='GT' if not _gt_legend_shown else None,
        showlegend=not _gt_legend_shown,
        legendgroup='gt',
        hovertext=_gb['name'],
        hoverinfo='text',
    ))
    _gt_legend_shown = True

# ── Ego marker ────────────────────────────────────────────────────────────────
obb_traces.append(go.Scatter(
    x=[0], y=[0], mode='markers',
    marker=dict(symbol='arrow-up', angle=90, size=16,
                color='yellow', line=dict(color='black', width=1)),
    name='ego',
))

mode_str = 'B1 (LiDAR tz corrected)' if B1_LIDAR_CORRECTION else 'Baseline'
fig_obb = go.Figure(data=obb_traces)
fig_obb.update_layout(
    title=f'OBB footprints — BEV, ego frame  |  {mode_str}  |  scene {SCENE_IDX} frame {FRAME_IDX}',
    xaxis=dict(title='X ego (forward, m)', range=[-10, 70],
               scaleanchor='y', scaleratio=1,
               showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False),
    yaxis=dict(title='Y ego (left, m)', range=[-35, 35],
               showgrid=True, gridcolor='rgba(255,255,255,0.08)', zeroline=False),
    annotations=obb_annotations,
    plot_bgcolor='rgb(15,15,25)', paper_bgcolor='rgb(15,15,25)',
    font=dict(color='white'),
    legend=dict(bgcolor='rgba(0,0,0,0.55)', font=dict(size=10)),
    width=1000, height=580, margin=dict(l=0, r=0, t=40, b=0),
)
fig_obb.show()


In [ ]:
print(f'Scene {SCENE_IDX}, frame {FRAME_IDX} — {CAMERA}')
print(f'B1_LIDAR_CORRECTION = {B1_LIDAR_CORRECTION}')
print(f'B2_GROUND_ANCHORING = {B2_GROUND_ANCHORING}')
print()
print(f'{"Ped":<5} {"tz [m]":<10} {"b1_mode":<30} {"b2_mode"}')
print('-' * 80)
for i, r in enumerate(body_results):
    tz  = float(r['cam_t'][2])
    b1  = r.get('b1_mode', '—')
    b2  = r.get('b2_mode', '—')
    print(f'{i:<5} {tz:<10.2f} {b1:<30} {b2}')
